## EV333 Problem Set 3: Coupled Climate Variability

## Overview 

In this problem set, you will demonstrate both your qualitative and quantitative understanding of this week's course material and use Python to explore coupled atmopshere interactions with the ocean. The El Niño–Southern Oscillation (ENSO) is an important coupled ocean‐atmosphere climate phenomenon with global impacts on temperature, sea level pressure, and precipitation patterns (Bjerknes, 1969; Ropelewski & Halpert, 1987). This assignment explores observed ENSO variability from 1950-Present. 

You will use the ERA5 atmopsheric reanalysis product produced by the Copernicus Climate Change Service at the European Centre for Medium-Range Weather Forecasts (ECMWF). The reanalysis combines model data with observations from around the world to provide hourly estimates of a large number of atmospheric, land, and oceanic climate variables. The data was accessed from the [ECMWF Climate Data Portal](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5).

To foster a collaborative learning environment, you will work on this assignment with a partner. The pairings will be randomly determined and posted on Canvas. Each person is required to complete and contribute to all questions. This includes performing all calculations/analyses and writing your own code. *It is against the honor code to divide up the questions among different individuals or use generative AI/chatbots on any aspect this assignment. All answers must be written in your own words.* All figures will be generated in this Jupyter Notebook. Please use comments (`#`) to organize your code and make it more readable. **This problem set is worth 75 points.**

### Learning Outcomes
The goal of this is assignment is to demonstrate your understanding of fluid motion in a rotating reference frame and the atmopsheric general circulation. By completing this problem set you will: 

- Articulate how and why the Pacific Walker Circulation varies during ENSO events
- Identify global temperature, rainfall, and pressure patterns associated with historical ENSO events
- Link the global anomaly patterns with key thermodynamic and dynamic concepts
- Manipulate gridded atmospheric reanalysis data using the Python Xarray package and perform statistical analyses, including calculating climatology-removed monthly anomalies
- Generate high-quality maps of monthly anomalies using Cartopy, Matplotlib and cmocean
- Develop a set of sample code that you can later build upon for your final project.

### Canvas Submission
1. **Jupyter Notebook (individual submission):** Run your Jupyter Notebook from the beginning to confirm that all figures generate correctly and that there are no error messages. Please name your notebook as: LastName_FirstName_EV333_PS3.ipynb and upload your file to Canvas. All group members must submit their code individually. 
2. **Synthesis Questions (group submission):** Write your responses to the short-answer questions in a Microsoft Word or shared Google document. Include all relevant figures. Export your final write-up as a PDF and upload it to Canvas. Only one member of your group needs to submit your responses.  

Out of fairness to all students, assignments submitted after the deadline will receive an automatic 10% deduction per day unless you make arrangements with Professor Lawman well in advance of the deadline.

---
## Datasets

**Required data:** (Download from GitHub)
- **Nino3.4 sea surface temperature anomaly:** detrend.nino34.monthly.txt
- **ERA5 Sea surface temperature:** ERA5_monthly_sst_regrid.nc 
- **ERA5 Mean sea level pressure:** ERA5_monthly_msl_regrid.nc
- **ERA5 Total precipitation:** ERA5_monthly_tp_regrid.nc 

### Niño3.4 Monthly SST Anomalies

Sea surface temperature anomalies (SSTA) averaged across the Niño 3.4 region (5°N-5°S, 120°-170°W) in the central equatorial Pacific are used to define El Niño (warm phase) and La Niña (cool phase) events. Monthly SST anomalies from January 1950- February 2024 are provided in `detrend.nino34.monthly.txt`. The anomalies are based on centered 30-year reference periods updated every 5 years (i.e., sliding climatologies). This approach detrends the data, removing the long-term observed warming signal. Data downloaded from [NOAA/CPC](https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/detrend.nino34.ascii.txt).


### ERA5 Atmospheric Reanalysis Product

ERA5 is produced by the Copernicus Climate Change Service at the European Centre for Medium-Range Weather Forecasts (ECMWF). The data was accessed from the [ECMWF Climate Data Portal](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5). The full reanalysis product covers the period from January 1940 to present. The original monthly datasets were regridded to a much coarser  ~4 $^\circ$ latitude x 4$^\circ$ longitude horizontal resolution (16.7 MB per file). The 4$^\circ$ resolution will still show the broad global patterns. You will work with sea surface temperature (SST) and mean sea level pressure (SLP) data in this lab.

**Sea surface temperature (K):** This parameter (SST) is the temperature of sea water near the surface. This parameter has units of kelvin (K).

**Mean sea level pressure (Pa):** This parameter is the pressure (force per unit area) of the atmosphere at the surface of the Earth, adjusted to the height of mean sea level. It is a measure of the weight that all the air in a column vertically above a point on the Earth's surface would have, if the point were located at mean sea level. It is calculated over all surfaces - land, sea and inland water. Maps of mean sea level pressure are used to identify the locations of low and high pressure. Contours of mean sea level pressure also indicate the strength of the wind. Tightly packed contours show stronger winds. The units of this parameter are pascals (Pa). Mean sea level pressure is often measured in hPa and sometimes is presented in units of millibars, mb (1 hPa = 1 mb = 100 Pa).

**Total precipitation (m)**: This parameter is the accumulated liquid and frozen water, comprising rain and snow, that falls to the Earth's surface. It is the sum of large-scale precipitation and convective precipitation. Large-scale precipitation is generated by the cloud scheme in the ECMWF Integrated Forecasting System (IFS). The cloud scheme represents the formation and dissipation of clouds and large-scale precipitation due to changes in atmospheric quantities (such as pressure, temperature and moisture) predicted directly by the IFS at spatial scales of the grid box or larger. Convective precipitation is generated by the convection scheme in the IFS, which represents convection at spatial scales smaller than the grid box. This parameter does not include fog, dew or the precipitation that evaporates in the atmosphere before it lands at the surface of the Earth. This parameter is accumulated over a particular time period which depends on the data extracted. ***For the monthly averaged reanalysis, the accumulation period is 1 day.***

---

In [ ]:
# import Python packages
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cmocean
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

# Part 1. Load & plot observed Niño3.4 monthly SSTA

The NOAA/CPC monthly Niño 3.4 detrended sea surface temperature anomalies (SSTA) have already been calculated for you. The effect of the seasons was removed when calculating monthly SSTA. 

**<span style='color:Red'> [1pt] Insert a cell below (`+`).  </span> Define a variable called `fileName`** specifying the full file path for the `detrend.nino34.monthly.txt` dataset that contains the monthly Niño 3.4 SST anomalies:

```
fileName = '<add the full path to the file here>'
```

**Load the file as a Pandas dataframe called `df` by running the cell below.**

The learning goals for this assignment focus on practice manipulating, plotting, and interpreting the data. To save you a lot of time, the pre-processing steps are completed for you in the cell below. The resulting, nicely formatted Pandas DataFrame `df` is displayed at the end. Review the `df` output. 

In [ ]:
# load data as a Pandas DataFrame (df)
df = pd.read_fwf(fileName)

# remove data that is not needed for this assignment
df = df.drop(['TOTAL','ClimAdjust'], axis=1)

# convert to datetime
time = pd.to_datetime(dict(year=df.YR, month=df.MON, day=1))
df = df.drop(['MON'], axis=1)
df = df.rename(columns={'YR': 'time'})
df['time'] = time

# display the DataFrame
df

Next let's define a variable called `df_smooth` and calculate the 5-month running mean. This step has been done for you in the cell below and the output is displayed.

In [ ]:
df_smooth = df.ANOM.rolling(window=5, center=True).mean().to_frame()
df_smooth.insert(0,'time',df.time)
df_smooth = df_smooth.dropna()
df_smooth

**<span style='color:Red'>  [4pts, 1pt each] Familiarize yourself with the data.</span> Working with your group, answer the questions below in a separate document.** 

1. What is the Niño3.4 region? Why is it used to study ENSO?
2. The provided dataset provides *linearly detrended* monthly sea surface temperature anomalies (SSTA). Why is it necessary to detrend the original SSTs prior to calculating the SST anomalies? 
3. What are seasonal-cycle removed monthly SSTAs? Why is it important to remove the effect of the seasons in order to understand ENSO behavior?
4. What are the benefits of calculating a 5-month running mean?  

### Generate a plot of Niño 3.4 SSTA versus time

Each column in a **DataFrame** is a **Series**. To select a single column, use square brackets `[]` with the name of the column of interest as a string. This is very similar to how you accessed an xarray DataArray from a DataSet in the previous lab. 

For example: `ssta = df['ANOM']` where `df` is the DataFrame and `ANOM` is the name of a variable in the DataFrame. Both single and double quotes will work for strings.

**<span style='color:Red'> [4pts] Insert a cell below (`+`) and generate an x, y line plot of Niño3.4 SSTA vs. time:** </span> 
1. Plot the monthly data as a red line and the 5-month running mean (smoothed) data as a thicker black line.
2. Add a horizontal line at 0 deg C to help the viewer easily differentiate positive and negative anomalies. This is done with the code: `plt.axhline(y = 0, color = 'k', linestyle = '-')`
3. Add a title and axes labels (with units!) to your plot.
4. Add a legend to differentiate the red and black lines. 
5. Save the figure as `Nino34_SSTA.png`.

**Some sample code is provided below for you to modify.** *Tip! Revisit your climate model problem set and the Python Intro notebook if you need a plotting refresher.* 

```
# define figure
plt.figure(figsize=[12,4])

# add a horizontal line at 0 deg C
plt.axhline(y = 0, color = 'k', linestyle = '-') 

# plot Nino34 SSTA vs. time (monthly)
...add your code here...

# plot Nino34 SSTA vs. time (5-month running mean)
...add your code here...

# add a title, axes labels (with units), and a legend
...add your code here...

# save the figure
plt.savefig('Nino34_SSTA.png', bbox_inches='tight')

```

# Part 2: Long-term mean sea-surface temperature (SST)

Part 2 builds upon the skills you developed in problem set 2, but with a new SST dataset.

1. Load the ERA5 SST data using the Python xarray package and define it as a variable `SST`
2. Subset the data to January 1950-Present. *Hint use the `.isel()` method.*
3. Extract the `'sst'` DataArray from the DataSet and convert the temperature values from kelvin to degrees Celsius. *Hint: We can access `sst` by treating our DataSet like a dictionary: `SST['sst']`.*
4. Display the output to make sure you analyzed the data correctly.

**<span style='color:Red'> [3pts] Insert a cell below (`+`). Load, subset, and convert the SST data following the steps above.** </span> 

Now calculate the long-term mean SST and generate a Pacific-centered global map.

**<span style='color:Red'> [1pt] Insert a cell below, define a variable `m_SST`, and calculate the long-term mean SST.**

Recall for plotting maps using Cartopy we need to specific a map projection, a colormap variable from the cmocean package, and the levels for the colorbar. Example code is provided below, but please consult your previous problem set for additional examples. 

```
# map projection
proj = ccrs.Robinson(central_longitude=180)

# selected color map from cmocean colormaps for oceanography
cmap = ...add your code here...

# NumPy array for the color bar levels, adjust range and increment depending on the dataset
lev = np.arange(...set your values here...);
```

**<span style='color:Red'> [1pt] Insert a cell below (`+`) to specify the mapping variables (`proj`, `cmap`, and `lev`).**

**<span style='color:Red'> [3pts] Insert a cell below (`+`) Generate a map of mean SST using the xarray `contourf.()` method.** 

For full credit, add coastlines, latitude and longitude gridlines, and a title. The colorbar label should include the units. Since we're plotting only ocean data, fill in the coastlines with a lightgray color (sample code provided). Save the figure as `ERA5_mean_SST.png`

Some sample code is provided below for you to modify: 

```
# define figure and axes, figure size, and resolution (300 dpi))
fig = plt.figure(figsize=(9, 4.5), dpi=300)
ax = plt.axes(projection = proj)

# filled contour map of mean temperature
...add your code here...

# add coastlines
...add your code here...

# add coastlines masking the land in lightgray
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '110m', edgecolor='k', facecolor='lightgray'))

# add grid lines
...add your code here...

# add title
ax.set_title("UPDATE TITLE")

# save figure 
fig.savefig('ERA5_mean_SST.png', facecolor = 'white', transparent = False, bbox_inches ='tight')
```

# Part 3: Long-term mean sea level pressure (SLP) and precipitation

Repeat the steps in Part 2 using the ERA5 sea level pressure dataset:

1. Define a variable `SLP` and load the ERA5 mean sea level ('msl') pressure data using the Python xarray package
2. Subset the data to January 1950-Present
3. Convert from Pa to hPa. We can access the `msl` variable by treating our DataSet like a dictionary: `SLP['msl']`.
4. Define a variable `m_slp` and calculate the long-term mean
5. Generate a global map, similar to SST. Use an appropriate cmocean colormap.
6. Format the map by adding a title, axes labels with units, coastlines, etc. **Do not** fill in the coastlines like with SST.
7. Save the figure as `ERA5_mean_slp.png`

**<span style='color:Red'> [4pts] Insert cell(s) below (`+`) and generate a map of the long-term mean sea level pressure.**

Now load the ERA5 total precipitation data and process the data the same way as SST and SLP. You **do not** need to generate a global map since you already have this from Problem Set 2. 


**<span style='color:Red'> [1pts] Insert a cell below (`+`) to:**

1. Define a variable `PRECIP` and load the ERA5 total precipitation ('tp') data using the Python xarray package
2. Subset the data to January 1950-Present
3. Convert the `tp` DataArray from m/day to mm/day.
4. Display the output to make sure you have processed the data correctly

## Part 4: Calculate climatology-removed monthly SST anomalies (SSTA) 

In this assignment, we want to *remove* the seasonal cycle from the SST, SLP, and precipitation data to yield monthly anomalies. But first we need to remove the long-term warming trend from the data. Revisit your synthesis questions to think about why this is the case.

The cell below contains a function called `detrend` that will by default remove a linear trend from a **DataArray**. You do not need to modify the function. Run the cell below.

In [ ]:
#---do not modify---
def detrend(da, dim, deg=1):
    # detrend along a single dimension
    p = da.polyfit(dim=dim, deg=deg)
    fit = xr.polyval(da[dim], p.polyfit_coefficients)
    return da - fit

**<span style='color:Red'> [1pt] Insert a cell below (`+`)** and use the detrend function to linearly detrend the gridded SST DataArray (`deg = 1`) you named above as `SST`. Call the variable `SST_Detrend`.

In this problem set you will calculate an average seasonal cycle, also known as the climatology. This involved calculating the average January value, the average February value, the average March value, ..., etc. 

This will be accomplished using the [groupby](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html) operation that involves splitting the object, applying a function, and combining the results. This is useful for grouping large amounts of data and computing operations on these groups. To group a DataArray, use the syntax: `ds.groupby(ds['Year_Month'].dt.month)` where `ds` is a generic variable name meant to represent the DataArray. The line of code says to group the data by `month`. Then you can calculate the mean by performing the `.mean()` method on the grouped DataArray. 

**<span style='color:Red'> [3pts] Insert a cell below (`+`)** and calculate the climatology-removed monthly SST anomalies.

Using the detrended data `SST_Detrend`, calculate the climatology-removed monthly SSTA as a variable called `anom_SST`. To do this:

1. You will first need to calculate the climatology (average seasonal cycle). The groupby operation is demonstrated in the example code below. Name the average seasonal cycle as a variable called `clim_SST.`
2. Define a variable `anom_SST` that first groups the detrended data `SST_Detrend` by month, and then subtracts the climatology, `clim_SST` to yield monthly SST anomalies.
3. Display `anom_SST` to check the dimensions.

Sample code: 
```
# climatology (average seasonal cycle) of the filtered data (do not modify)
clim_SST = SST_Detrend.groupby(SST_Detrend['time'].dt.month).mean()

# monthly anomalies
...add your code here...

# display the output DataArray
anom_SST
```

**<span style='color:Red'> [2pts] Knowledge check!** 

In plain language, what does the line `clim_SST = SST_Detrend.groupby(SST_Detrend['time'].dt.month).mean()` actually do in practice? What about the monthly anomaly calculation? **Answer this question in your group's shared lab document.**

# Part 5: Calculate climatology-removed monthly SLP and precipitation anomalies 

Repeat part 4 for SLP:
1. Detrend the data using the `detrend` function
2. Calculate the climatology as a variable called `clim_SLP`
3. Calculate the monthly anomalies as a variable called `anom_SLP`

Now repeat part 4 for PRECIP:
1. Detrend the data using the `detrend` function
2. Calculate the climatology as a variable called `clim_PRECIP`
3. Calculate the monthly anomalies as a variable called `anom_PRECIP`

**<span style='color:Red'> [6pts, 3 pts each] Insert cell(s) below (`+`)** and calculate the monthly anomalies for both variables.

# Part 6. Identify SST, SLP, and precipitation anomalies during historical ENSO events

Now, let's explore the observed patterns during El Niño and La Niña events! Using your Niño3.4 anomaly time series, look at the values during 1982-1983. ENSO events peak during Northern Hemisphere winter (e.g., November-January or December-February) and thus bridge calendar years. For example the 1982-83 El Niño would be November 1982-January 1983. 

**<span style='color:Red'> [4pts] Insert a cell below (`+`):**

1. Define two variables `start_time` and `end_time`. These variables will be assigned a datetime string following the convention `YYYY-MM-DD`. Let's start by subsetting the data to span the 1982-83 El Niño.

```
start_time = '1982-11-01'
end_time = '1983-01-31'
```

2. Then use the `.sel(time=slice())` method to subset the monthly SST, SLP, and precipitation anomalies to the time interval specified by `start_time` and `end_time`.

3. Then calculate the mean for the selected 3-month subset using the `mean.()` method.

**Example code is provided below as a guide:**

```
# calculate the average monthly anomalies during the selected ENSO event
# SSTA
ENSO_SSTA = anom_SST.sel(time=slice(<add your code here>)).mean(<add your code here>)

# SLP anomalies
ENSO_SLPA = <add your code here>

# precipitation anomalies
ENSO_PRECIPA = <add your code here>

```

**<span style='color:Red'> [6pts, 2 pts each map] Insert cell(s) below (`+`) to generate anomaly maps:**

Generate Pacific-centered maps of mean SSTA, SLP, and precipitation anomalies for the selected ENSO event. Specify a colormap appropriate for each type of anomalies and ensure that the colorbar is precisely centered on zero (no change). Adjust the colorbar levels for each plot such that your final maps are well-formatted and visually appealing. Include a title, axes labels with units, and lat/lon gridlines.  

Mask all the land masses with another color for the SSTA map only. Display all the data for the SLP anomaly map even over land.

Save your SSTA figure using the following code: 
```
fig.savefig('ERA5_sst_anom_' + start_time + '_' + end_time + '.png', facecolor = 'white', transparent = False, bbox_inches ='tight')
```

Save your SLP anomaly figure using the following code: 
```
fig.savefig('ERA5_slp_anom_' + start_time + '_' + end_time + '.png', facecolor = 'white', transparent = False, bbox_inches ='tight')
```

Save your precipitation anomaly figure using the following code: 
```
fig.savefig('ERA5_precip_anom_' + start_time + '_' + end_time + '.png', facecolor = 'white', transparent = False, bbox_inches ='tight')
```

This way you can easily find the desired maps based on their start/end dates.

# Part 7: Week 3 Synthesis Questions [31 pts]

When answering the following questions, please consult your Problem Set 3 figures, your class notes, and the assigned readings (textbook chapters, journal club articles, etc.). Please explain your reasoning for all questions to receive full credit.

**The synthesis questions below should be treated as cumulative assessment questions that demonstrate your collective knowledge of our Week 1 through 3 course material. When appropriate, please incorporate your knowledge of thermodynamics, static stability, fundamental forces, general atmospheric circulation, monsoons, the Pacific Walker Circulation, etc. to receive full credit.** Each person is required to complete and contribute to all questions.

1. [3pts] For the 1982-83 El Niño, please describe the global SST, SLP, and prcipitation anomaly patterns you observe. First describe the patterns in the equatorial Pacific (western, central, and eastern Pacific). Then select another 2-3 locations outside the tropical Pacific that you find interesting and describe the spatial patterns you observe. Include the 3 maps you generated above in your write-up. 
2. [6pts] How do the SST, SLP, and precipitation anomalies covary during the 1982-83 El Niño? To receive full credit, explain the *physical processes* and importantly *why* these variables are linked both thermodynmically and dynamically. 
3. [2pts] Why are Niño 3.4 SSTAs (plotted in this assignment) and the Southern Oscillation Index negatively correlated? To answer this question, draw on your knowledge of coupled ocean-atmosphere interactions, the mean climate of the equatorial Pacific, the Pacific Walker Circulation, and the tropical Pacific SST and SLP anomalies during ENSO events. 
4. [3pts, 1.5 pt each] Consult your Niño 3.4 time series plot. Re-run your code to generate SST, SLP, and precipitation anomaly maps for *at least* 2 other El Niño events. You should only change the start and ending years, while keeping the subset of months to boral winter (NDJ or DJF). Do not copy and paste any mapping code. How do the patterns compare? Comment on both the similarities and differences between the different events. Add all your figures to your shared word document. *Hint: If you are having trouble identifying an event from your figures, you may consult the [NOAA Oceanic Nino Index for inspiration](https://origin.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php).*
5. [2pts, 1pt each] Re-run the code to generate your boreal winter anomaly maps for *at least* 2 La Niña events. Only change the year values for the subset. How do the patterns compare between different La Niña events? What about La Niña versus El Niño? Add your figures to a word document and compare/contrast the results. 
6. [2pts] How do the magnitudes of the anomalies compare between warm phase and cool phase events? Which tend to yield larger anomalies in the Nino3.4 region? El Niño or La Niña?
7. [3 pts] How does a *weakning* of the Pacific Walker Circulation and the trade winds along the equator initiate a positive feedback loop involving sea surface temperatures and further reduction of the trade winds? What do we call this feedback loop?
8. [4pts] Let's examine some key atmospheric teleconnections! (a) How does the Aleutian Low in the North Pacific change during El Niño events (*Hint: consult both your mean and anomaly maps and reference them in your answer*)? How does this influence precipitation patterns along the west coast of North America? Use your knowledge of atmospheric circulation (fundamental forces) to explain why. (b) How does SLP and precipitation change in South America near the equator on the western versus eastern side of the Andes? From your maps, can you infer any ENSO-related changes to the South American summer monsoon? 
9. [2pts] How does the Pacific ITCZ shift during El Niño versus La Niña events. Consult your maps and feel free to annotate them if it helps you explain your reasoning.
10. [3pts] What are some of the other global impacts of ENSO? Your notes, the assigned readings, or the ENSO blog are all useful resources for answering this question. Please provide at least 3 specific examples of climate, societal, or other impacts on humans or other ecosystems. 

## Congratulations! You completed your final EV333 Problem Set! 

**To submit your assignment:**
1. Set the selected ENSO event back to the 1982-83 El Niño
2. Re-run the whole Notebook and check there are no errors
4. Submit your Jupyter Notebook (LastName_FirstName_EV333_PS3.ipynb) on Canvas (1 submission per person).
5. Compile your written responses and relevant figures in a separate Word or Google document. Export the document as a PDF and upload it to Canvas (1 submission per group).